In [1]:
import operator
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report

from deap import base, creator, tools, gp, algorithms

In [2]:
# Safe division to avoid division-by-zero errors
def safeDiv(a, b):
    return a / b if b != 0 else 1  # Prevents division by zero

In [3]:
# Conditional if operator
def if_operator(condition, output_true, output_false):
    return output_true if condition > 0 else output_false

In [4]:
# Classification function based on fixed thresholds (Algorithm 1)
def classify_output(y_pred, mode):
    y_pred = np.clip(y_pred, -100, 100)  # Clip to avoid overflow in sigmoid
    y_pred_sigmoid = 1.0 / (1.0 + np.exp(-y_pred))  # Sigmoid function

    if mode == "binary":
        return (y_pred_sigmoid > 0.5).astype(int)  # Binary classification threshold

    elif mode == "multiclass":
        thresholds = [0.20, 0.40, 0.60, 0.80]  # Multi-class thresholds
        return np.digitize(y_pred_sigmoid, thresholds)

    elif mode == "multistage":
        thresholds = [0.20, 0.80]  # Multi-stage thresholds
        return np.digitize(y_pred_sigmoid, thresholds)

In [5]:
from sklearn.utils.class_weight import compute_class_weight

# GP evaluation function following Algorithm 1
def evalGP(individual, X_train, y_train, mode):
    print("\n🚀 Evaluating GP Individual...\n")
    
    # Compile GP tree into a function f(x)
    func = gp.compile(expr=individual, pset=pset)

    # Apply f(x) to each instance in the training set
    y_pred = np.array([func(*x) for x in X_train])

    # Apply sigmoid to constrain outputs between 0 and 1
    y_pred_sigmoid = 1.0 / (1.0 + np.exp(-y_pred))

    # Classify outputs using fixed thresholds per mode
    y_pred_class = np.array([classify_output(val, mode) for val in y_pred_sigmoid])

    # Ensure predictions match valid class labels using y_train (not y_test)
    valid_classes = np.unique(y_train)  # Get valid classes from training labels
    y_pred_class = np.array([c if c in valid_classes else np.random.choice(valid_classes) for c in y_pred_class])

    # Compute class weights
    class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    weights_dict = {cls: weight for cls, weight in zip(np.unique(y_train), class_weights)}
    
    # Apply weights to accuracy calculation
    fitness = balanced_accuracy_score(y_train, y_pred_class, sample_weight=[weights_dict[y] for y in y_train])

    return (fitness,)

In [6]:
# Function to run GP evolution
def run_gp(X_train, y_train, mode, population_size=10, ngen=50, cxpb=0.8, mutpb=0.19):
    print(f"\n>> Starting GP Evolution for {mode.upper()} Classification...\n")
    
    # Initialize population
    pop = toolbox.population(n=population_size)
    hof = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)
    stats.register("max", np.max)

    print(f">> Population initialized with {population_size} individuals.")
    print(f">> Running evolution for {ngen} generations...\n")

    for gen in range(ngen):
        print(f">>> Generation {gen+1}/{ngen}")

        # Evaluate fitness of the population
        fitnesses = list(map(toolbox.evaluate, pop))
        for ind, fit in zip(pop, fitnesses):
            ind.fitness.values = fit

        # Print fitness of best individual in the generation
        best_ind = max(pop, key=lambda ind: ind.fitness.values[0])
        print(f">>> Best Individual (Gen {gen+1}): {best_ind}, Fitness: {best_ind.fitness.values[0]:.4f}")

        # Selection
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))

        # Apply crossover
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < cxpb:
                toolbox.mate(child1, child2)
                del child1.fitness.values, child2.fitness.values
        print(">>> Crossover Applied")

        # Apply mutation
        for mutant in offspring:
            if random.random() < mutpb:
                toolbox.mutate(mutant)
                del mutant.fitness.values
        print(">>> Mutation Applied")

        # Replace old population with new offspring
        pop[:] = offspring

        # Evaluate fitness again
        invalid_ind = [ind for ind in pop if not ind.fitness.valid]
        fitnesses = list(map(toolbox.evaluate, invalid_ind))
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        # Print best fitness in the generation
        best_gen_fitness = max(ind.fitness.values[0] for ind in pop)
        print(f">>> Best Fitness (Gen {gen+1}): {best_gen_fitness:.4f}")

    print("\n>> Evolution Completed!")

    # Retrieve best individual from Hall of Fame
    best_individual = hof[0] if len(hof) > 0 else max(pop, key=lambda ind: ind.fitness.values[0])
    print("\n>> Best GP Individual After Evolution:")
    print(best_individual)

    return best_individual, stats

In [7]:
# Plot confusion matrix using seaborn
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=np.unique(y_true), yticklabels=np.unique(y_true))
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(title)
    plt.show()

In [8]:
# Evaluate the best GP individual on test data and plot confusion matrix
def evaluate_best_model(best_individual, X_test, y_test, mode):
    func = gp.compile(expr=best_individual, pset=pset)
    y_pred = np.array([func(*x) for x in X_test])
    y_pred_sigmoid = 1.0 / (1.0 + np.exp(-y_pred))
    y_pred_class = np.array([classify_output(val, mode) for val in y_pred_sigmoid])
    acc = balanced_accuracy_score(y_test, y_pred_class)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_class))
    plot_confusion_matrix(y_test, y_pred_class, f"Confusion Matrix ({mode})")
    return acc

In [9]:
# Plot evolution progress over generations
def plot_evolution(stats, title):
    gens = range(len(stats))
    avg_acc = stats.select("avg")
    max_acc = stats.select("max")
    plt.figure(figsize=(8, 5))
    plt.plot(gens, avg_acc, label="Average Fitness", linestyle="--", marker="o")
    plt.plot(gens, max_acc, label="Max Fitness", marker="s")
    plt.xlabel("Generation")
    plt.ylabel("Balanced Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid()
    plt.show()

In [10]:
# Simple alert system for multistage classification
def alert_system(best_individual, X_test):
    func = gp.compile(expr=best_individual, pset=pset)
    y_pred = np.array([func(*x) for x in X_test])
    y_pred_sigmoid = 1.0 / (1.0 + np.exp(-y_pred))
    y_pred_class = np.array([classify_output(val, "multistage") for val in y_pred_sigmoid])
    alert_labels = {
        0: "Normal Traffic",
        1: "Reconnaissance Detected!",
        2: "Foothold Establishment / Lateral Movement Detected!",
        3: "Data Exfiltration Detected! Possible Breach!"
    }
    print("\nReal-Time Attack Detection Alerts (first 10 instances):")
    for i, cls in enumerate(y_pred_class[:10]):
        print(f"Alert {i+1}: {alert_labels.get(cls, 'Unknown')}")

In [ ]:
if __name__ == "__main__":
    print("\>> nStep 1: Loading and Preprocessing the Dataset...\n")

    # Load dataset
    df = pd.read_csv("../Datasets/dapt2020.csv")
    print(f">> Dataset Loaded! Shape: {df.shape}")

    # Drop unnecessary columns
    drop_features = [
        "Flow ID", "Src IP", "Dst IP", "Src Port", "Timestamp", 
        "Fwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags", "URG Flag Cnt", 
        "Fwd Byts/b Avg", "Fwd Pkts/b Avg", "Fwd Blk Rate Avg", 
        "Bwd Byts/b Avg", "Bwd Pkts/b Avg", "Bwd Blk Rate Avg", "Fwd Seg Size Min"
    ]
    df.drop(columns=drop_features, inplace=True, errors="ignore")
    print(f">> Dropped Unnecessary Features! Remaining Columns: {df.columns.tolist()}")

    # Create label columns
    print("\n>> Creating Label Columns...")
    df["Binary_Label"] = df["Stage"].apply(lambda x: 0 if x == "Benign" or "BENIGN" else 1)
    multiclass_mapping = {
        "Normal": 0, "Reconnaissance": 1, "Foothold Establishment": 2,
        "Lateral Movement": 3, "Data Exfiltration": 4
    }
    df["Multiclass_Label"] = df["Stage"].map(multiclass_mapping)
    
    multistage_mapping = {
        "Normal": 0, "Reconnaissance": 1, "Foothold Establishment": 2,
        "Lateral Movement": 2, "Data Exfiltration": 3
    }
    df["Multistage_Label"] = df["Stage"].map(multistage_mapping)
    print(">> Label Columns Created!")

    # Select numeric feature columns
    feature_cols = [col for col in df.columns if col not in ["Stage", "Binary_Label", "Multiclass_Label", "Multistage_Label"]]
    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")
    df.fillna(0, inplace=True)
    X = df[feature_cols].values.astype(float)
    print(f">> Feature Extraction Completed! Number of Features: {len(feature_cols)}")

    # Normalize features
    print("\n>> Normalizing Features...")
    scaler = MinMaxScaler()
    X = scaler.fit_transform(X)
    print(">> Normalization Completed!")

    # Split data
    print("\n>> Splitting Data into Training and Test Sets...")
    X_train, X_test, y_train_bin, y_test_bin = train_test_split(X, df["Binary_Label"].values, test_size=0.2, random_state=42)
    _, _, y_train_multi, y_test_multi = train_test_split(X, df["Multiclass_Label"].values, test_size=0.2, random_state=42)
    _, _, y_train_stage, y_test_stage = train_test_split(X, df["Multistage_Label"].values, test_size=0.2, random_state=42)
    print(f">> Data Splitting Completed! Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

    print("\n>> Step 2: Setting Up Genetic Programming (GP) with DEAP...\n")

    # GP setup
    pset = gp.PrimitiveSet("MAIN", X_train.shape[1])
    pset.addPrimitive(operator.add, 2)
    pset.addPrimitive(operator.sub, 2)
    pset.addPrimitive(operator.mul, 2)
    pset.addPrimitive(safeDiv, 2)
    pset.addPrimitive(lambda a, b: a if a > b else b, 2, name="max")
    pset.addPrimitive(lambda a, b: a if a < b else b, 2, name="min")

    for i in range(X_train.shape[1]):
        pset.renameArguments(**{f"ARG{i}": f"X{i}"})

    # Delete previous GP classes if they exist
    if "FitnessMax" in creator.__dict__:
        del creator.FitnessMax
    if "Individual" in creator.__dict__:
        del creator.Individual

    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax)

    toolbox = base.Toolbox()
    toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=2, max_=6)
    toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("mate", gp.cxOnePoint)
    toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr, pset=pset)
    toolbox.register("select", tools.selTournament, tournsize=7)

    print(">> Genetic Programming Setup Completed!")

    # Run GP for Multi-class Classification
    print("\n>> Running GP Evolution for Multi-class Classification...")
    # toolbox.unregister("evaluate")
    toolbox.register("evaluate", evalGP, X_train=X_train, y_train=y_train_multi, mode="multiclass")
    best_multiclass, stats_multiclass = run_gp(X_train, y_train_multi, mode="multiclass")
    print("\n>> Best GP Individual for Multi-class Classification:")
    print(best_multiclass)

    # Run GP for Multi-stage Classification
    print("\n>> Running GP Evolution for Multi-stage Classification...")
    toolbox.unregister("evaluate")
    toolbox.register("evaluate", evalGP, X_train=X_train, y_train=y_train_stage, mode="multistage")
    best_multistage, stats_multistage = run_gp(X_train, y_train_stage, mode="multistage")
    print("\n>> Best GP Individual for Multi-stage Classification:")
    print(best_multistage)

    print("\n>> Step 4: Evaluating the Evolved Models on Test Data...\n")

    print("\n>> Evaluating Multi-class Classification Model...")
    acc_multiclass = evaluate_best_model(best_multiclass, X_test, y_test_multi, "multiclass")
    print(f">> Multi-class Classification Balanced Accuracy: {acc_multiclass:.4f}")

    print("\n>> Evaluating Multi-stage Classification Model...")
    acc_multistage = evaluate_best_model(best_multistage, X_test, y_test_stage, "multistage")
    print(f">> Multi-stage Classification Balanced Accuracy: {acc_multistage:.4f}")

    print("\n>> Step 5: Running Simple Alert System (for Multi-stage Detection)...\n")
    alert_system(best_multistage, X_test)

    print("\n>> All Steps Completed Successfully! <<")

\>> nStep 1: Loading and Preprocessing the Dataset...

>> Dataset Loaded! Shape: (86690, 81)
>> Dropped Unnecessary Features! Remaining Columns: ['Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Bwd PSH Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count

/tmp/ipykernel_15471/2241828608.py:14: RuntimeWarning: overflow encountered in exp
  y_pred_sigmoid = 1.0 / (1.0 + np.exp(-y_pred))



🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...



/tmp/ipykernel_15471/2241828608.py:14: RuntimeWarning: overflow encountered in exp
  y_pred_sigmoid = 1.0 / (1.0 + np.exp(-y_pred))



🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...

>>> Best Individual (Gen 1): max(mul(add(min(add(X29, X33), min(X68, X51)), min(safeDiv(X5, X7), add(X58, X71))), sub(safeDiv(mul(X58, X30), min(X0, X73)), max(mul(X16, X53), add(X64, X58)))), max(min(min(safeDiv(X12, X12), min(X70, X11)), add(mul(X50, X11), min(X66, X21))), add(sub(sub(X42, X60), mul(X67, X8)), max(add(X40, X14), mul(X66, X54))))), Fitness: 0.2500
>>> Crossover Applied
>>> Mutation Applied

🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...

>>> Best Fitness (Gen 1): 0.2500
>>> Generation 2/50

🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individ

...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...

>>> Best Individual (Gen 17): sub(X69, max(max(X32, min(sub(X64, X47), add(X9, X47))), max(X32, X0))), Fitness: 0.4301
>>> Crossover Applied
>>> Mutation Applied

🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...

>>> Best Fitness (Gen 17): 0.3507
>>> Generation 18/50

🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating G

/tmp/ipykernel_15471/2241828608.py:14: RuntimeWarning: overflow encountered in exp
  y_pred_sigmoid = 1.0 / (1.0 + np.exp(-y_pred))



🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...

>>> Best Fitness (Gen 26): 0.3722
>>> Generation 27/50

🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...



/tmp/ipykernel_15471/2241828608.py:14: RuntimeWarning: overflow encountered in exp
  y_pred_sigmoid = 1.0 / (1.0 + np.exp(-y_pred))



🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...

>>> Best Individual (Gen 27): sub(X32, max(max(X32, min(X67, add(max(max(add(X9, X32), min(sub(X0, X47), add(X32, X47))), max(X32, X0)), X47))), max(X32, X0))), Fitness: 0.3916
>>> Crossover Applied
>>> Mutation Applied

🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...


🚀 Evaluating GP Individual...

